# Non-linear (XGBoost) destination probe -- run all cells top to bottom

**What this is for.** The paper reports a *linear* probe inside SmolVLA's action expert. It
decodes the named destination on one checkpoint and fails on the other. A linear null has two
readings that mean very different things:

* the destination is **not encoded** at that site, or
* it **is** encoded, but not in a form a hyperplane can read.

The paper cannot currently separate them. This run fits gradient-boosted trees at every site on
the same split, same standardisation, same accuracy readout. Only the hypothesis class changes,
so any difference is attributable to linearity and nothing else.

**Reading the result** (the last cell prints it):

* boosted approximately equals linear -> the linear null is a **genuine absence**. This is the
  expected outcome and it strengthens the paper.
* boosted clearly beats linear on expert sites -> the destination is **present but not linearly
  readable**. That is a different claim from the one the paper makes, and a more interesting one.

Either outcome is useful. Nothing here should be tuned to produce a particular answer.

**Cost.** Roughly 30-60 minutes for both checkpoints on one GPU. Almost all of it is forward
passes; the probe fits are cheap. Activations are cached to disk, so any later probe variant
costs seconds instead of a full rerun.

**Run artifacts.** `scripts/run_probe.py` writes a resolved config and metrics for each run.
Activation caches stay local by design; review and commit the small result artifacts through
the repository's normal Git workflow after the run. This notebook never changes remotes,
credentials, or Git identity.

## 1. Confirm the accelerator and choose execution mode

This notebook defaults to **validation mode**: it installs the project into the active
kernel, tests the XGBoost control, and checks that all experiment inputs are wired correctly,
but it does not start a long model run. Set `RUN_FULL_XGBOOST_PROBE=1` before launching the
kernel only when you have reviewed the cost and have an Apple Metal (MPS) or CUDA GPU
available. Set `XGBOOST_PROBE_DEVICE=mps` to require Apple Metal, `cuda` to require CUDA,
or leave it unset to select an available accelerator automatically.

In [ ]:
import os
import shutil
import subprocess

RUN_FULL_PROBE = os.environ.get("RUN_FULL_XGBOOST_PROBE") == "1"
REQUESTED_DEVICE = os.environ.get("XGBOOST_PROBE_DEVICE", "auto").lower()
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)
else:
    print("nvidia-smi is unavailable; the install cell will report PyTorch accelerator status.")
print("execution mode:", "full probe" if RUN_FULL_PROBE else "validation")
print("requested device:", REQUESTED_DEVICE)

## 2. Find or clone the repository

Works whether you already cloned it or are starting cold. If this notebook already lives inside
the repository, it walks up to the root rather than nesting a second copy. It does not
checkout, pull, or otherwise alter an existing working tree.

In [ ]:
import os
import pathlib
import subprocess

REPO = "https://github.com/mzkaell/vla-where-does-language-die.git"


def find_repo_root(start: pathlib.Path):
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "src" / "models").exists():
            return d
    return None


root = find_repo_root(pathlib.Path.cwd())
if root is None:
    target = pathlib.Path.home() / "vla-where-does-language-die"
    if target.exists():
        raise RuntimeError(f"{target} exists but is not this repository; open the notebook from its clone.")
    subprocess.run(["git", "clone", REPO, str(target)], check=True)
    root = target

os.chdir(root)
print("repo root:", root)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True, check=True).stdout)
status = subprocess.run(["git", "status", "--short"], capture_output=True, text=True, check=True)
print(status.stdout or "working tree: clean")

## 3. Install the project into this notebook's kernel

`xgboost` is a core dependency in `pyproject.toml`; this installs the same project definition
used by the CLI and tests, using the Python interpreter that runs this notebook. Validation
mode installs only the base and test dependencies; a full GPU run also installs the VLA extra.

In [ ]:
import json
import sys

venv_python = pathlib.Path(root) / ".venv" / "bin" / "python"
PROJECT_PYTHON = str(venv_python if venv_python.exists() else pathlib.Path(sys.executable))
print("project python:", PROJECT_PYTHON)

extras = ".[vla,dev]" if RUN_FULL_PROBE else ".[dev]"
subprocess.run([PROJECT_PYTHON, "-m", "pip", "install", "-q", "-e", extras], check=True)

probe = subprocess.run(
    [
        PROJECT_PYTHON, "-c",
        "import json, importlib.metadata as im; import torch; "
        "print(json.dumps({"
        "'torch': torch.__version__, "
        "'mps': torch.backends.mps.is_available(), "
        "'cuda': torch.cuda.is_available(), "
        "'cuda_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "
        "'xgboost': im.version('xgboost')"
        "}))",
    ],
    capture_output=True, text=True, check=True,
)
device_info = json.loads(probe.stdout)

available_devices = {
    "mps": device_info["mps"],
    "cuda": device_info["cuda"],
}
if REQUESTED_DEVICE == "auto":
    DEVICE = next((name for name in ("mps", "cuda") if available_devices[name]), None)
elif REQUESTED_DEVICE in available_devices:
    DEVICE = REQUESTED_DEVICE if available_devices[REQUESTED_DEVICE] else None
else:
    raise ValueError(f"XGBOOST_PROBE_DEVICE must be auto, mps, or cuda; got {REQUESTED_DEVICE!r}")

print("torch", device_info["torch"], "| mps", available_devices["mps"], "| cuda", available_devices["cuda"])
print("xgboost", device_info["xgboost"])
if DEVICE == "cuda":
    print(device_info["cuda_name"])
elif DEVICE == "mps":
    print("Apple Metal (MPS) selected")
else:
    print("NO SUPPORTED ACCELERATOR VISIBLE")
if RUN_FULL_PROBE and DEVICE is None:
    raise RuntimeError("RUN_FULL_XGBOOST_PROBE=1 requires an MPS or CUDA GPU; refusing a multi-hour CPU fallback.")

## 4. Sanity-check the probe before a long run

These are the tests that make a null result meaningful. The boosted probe must recover
XOR-structured labels a linear probe cannot see, and stay near chance on shuffled labels.

In [ ]:
subprocess.run(
    [PROJECT_PYTHON, "-m", "pytest", "-q", "-p", "no:warnings", "tests/test_stats.py", "-k", "nonlinear"],
    check=True,
)

## 5. Launch both checkpoints sequentially in the background

The two runs share one accelerator, so they are deliberately sequential. The required LIBERO
demonstrations are fetched before launch (about 3.3 GB on a fresh machine). A detached process keeps
running after a notebook disconnect. The cell resumes safely by skipping a run only after its
`metrics.json` exists; it neither commits nor pushes results.

`--cache-activations` writes pooled activations to disk. The forward passes are the entire cost
and are identical for every probe family, so this makes follow-up probes nearly free.

In [ ]:
from pathlib import Path
import shlex

log_path = Path(root) / "xgb_probe.log"
run_specs = [
    ("k1000dai/smolvla_libero_finetune", "probe_nl_finetune"),
    ("k1000dai/smolvla_libero_scratch_80k", "probe_nl_scratch_80k"),
]

if not RUN_FULL_PROBE:
    print("Validation mode: full probe not launched. Set RUN_FULL_XGBOOST_PROBE=1 before starting the kernel on an MPS or CUDA machine.")
elif DEVICE is None:
    raise RuntimeError("An MPS or CUDA GPU is required for the full probe.")
else:
    subprocess.run([PROJECT_PYTHON, "scripts/download_data.py"], check=True)
    commands = []
    for checkpoint, run_id in run_specs:
        metrics = Path(root) / "results" / run_id / "metrics.json"
        if metrics.exists():
            commands.append(f"echo '== {run_id} already done, skipping'")
            continue
        args = [
            PROJECT_PYTHON, "-u", "scripts/run_probe.py", "--checkpoint", checkpoint,
            "--n-states", "150", "--device", DEVICE, "--nonlinear",
            "--cache-activations", "--run-id", run_id,
        ]
        commands.extend((f"echo '== {run_id} starting at $(date)'", shlex.join(args)))
    if not commands:
        print("Both nonlinear-probe results already exist.")
    else:
        with log_path.open("a", encoding="utf-8") as log:
            process = subprocess.Popen(
                ["/bin/bash", "-lc", "set -euo pipefail; " + "; ".join(commands)],
                cwd=root,
                stdout=log,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )
        print(f"launched PID {process.pid}; monitor {log_path}")

## 6. Monitor (re-run this cell whenever)

In [ ]:
if log_path.exists():
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]))
else:
    print("No log yet. In validation mode this is expected.")

## 7. Result status

In [ ]:
for _, run_id in run_specs:
    metrics = Path(root) / "results" / run_id / "metrics.json"
    print(f"{run_id}: {'complete' if metrics.exists() else 'not finished'}")

## 8. Detailed result summary

This prints the comparison the experiment exists to make. It never infers a result from an
unfinished run.

In [ ]:
import json
import pathlib

import numpy as np

for name in ["probe_nl_finetune", "probe_nl_scratch_80k"]:
    p = pathlib.Path("results") / name / "metrics.json"
    if not p.exists():
        print(f"{name}: not finished yet")
        continue
    d = json.loads(p.read_text())
    if "nonlinear" not in d:
        print(f"{name}: no nonlinear block -- was --nonlinear passed?")
        continue
    nl = d["nonlinear"]
    exp = [s for s in d["sites"] if s["site"].startswith("expert.")]
    deltas = [nl[s["site"]]["acc_novel"] - s["acc_novel"] for s in exp]
    gained = [s for s, dd in zip(exp, deltas) if dd > 0.10]
    # A boosted probe that gains only where its OWN shuffled control also rises is fitting
    # noise, not finding structure. Report both so one cannot be mistaken for the other.
    real = [s for s in gained if nl[s["site"]]["acc_shuffled"] < 0.35]
    print(f"===== {name} =====")
    print(f"  chance                           : {exp[0]['chance']:.3f}")
    print(f"  expert sites                     : {len(exp)}")
    print(f"  linear  max acc (novel)          : {max(s['acc_novel'] for s in exp):.3f}")
    print(f"  boosted max acc (novel)          : {max(nl[s['site']]['acc_novel'] for s in exp):.3f}")
    print(f"  mean(boosted - linear)           : {np.mean(deltas):+.3f}")
    print(f"  sites where boosted beats by .10 : {len(gained)}"
          f"  (of which shuffled stays low: {len(real)})")
    print("  ->", "LINEAR NULL IS GENUINE" if len(real) <= 2 else
          "NON-LINEARLY DECODABLE -- flag this, it changes a claim")
    print()

## 9. Optional download bundle

If you need to move finished artifacts from a temporary machine, create a local archive below.
The archive is stored under `results/`, where generated artifacts are ignored by Git.

In [ ]:
import tarfile

finished = [
    Path(root) / "results" / run_id
    for _, run_id in run_specs
    if (Path(root) / "results" / run_id / "metrics.json").exists()
]
if not finished:
    print("No completed nonlinear-probe results to bundle.")
else:
    archive = Path(root) / "results" / "xgb_results.tar.gz"
    with tarfile.open(archive, "w:gz") as bundle:
        for result_dir in finished:
            bundle.add(result_dir, arcname=result_dir.name)
    print(f"wrote {archive} ({archive.stat().st_size / 1_000_000:.1f} MB)")